# 3. Idealista: Detalle de Propiedades
**Propósito**: con este notebook se escrapean las páginas de las ofertas de los diferentes municipios, entramos dentro de cada oferta y recuperaos información como por ejemplo el comentario del vendedor, en ese comentario hay una gran cantidad información que después querremos explotar. Para cada provincia, se recorren todos las ofertas antes recuperadas y vamos recuperando oferta a oferta, esta recuperación se hace por baches, se ejecutan **N** anuncios de forma asincrona para reducir el tiempo de ejecución y aprovechar al máximo los ratios de llamadas concurrentes a la API de **ScrapFly**, así mismo si el trabajo se parase u ocurre cualquier problema durante la ejecución se aprovecha el avance conseguido lanzando, recuperando y guardando la información por baches. 

In [1]:
import sys
import time
import json
import math
import traceback
import pandas as pd
from datetime import datetime
from delta.tables import DeltaTable
from pyspark.sql import functions as F

sys.path.insert(0, '/tfm/python_notebooks')
from tfm_lib import idealista_scrap as idealista
from tfm_lib import utils as tfm_utils

In [2]:
debug_mode = False # True: modo TEST de Scrapfly (sin coste), 1 URL por municipio
provincia  = 'murcia'

# Número de anuncios que se envían a Scrapfly en paralelo por cada lote
BATCH_SIZE=60
SLEEP_BETWEEN_BATCHES=0.1  # Sin pausa entre lotes para máxima velocidad

# Control de reintentos:
# - RETRY_ERRORS=True: reintentar URLs con estado 'error' (errores temporales)
# - RETRY_ERRORS=False: no reintentar ningún URL ya procesado
# Nota: Los 'ok' y 'no_encontrado' NUNCA se reintentan
RETRY_ERRORS = False

OFERTAS_TABLE = 'raw.idealista.municipios_ofertas'
DETALLE_TABLE = 'raw.idealista.detalle_propiedades'

spark = tfm_utils.get_spark_session(f'Idealista_Detalle_{provincia}')

detalle_path = tfm_utils.full_table_path(DETALLE_TABLE)

print(f'[{datetime.now()}] Configuración cargada para: {provincia}')

[2026-05-26 22:21:57.038949] Configuración cargada para: murcia


## Carga Única de Datos
Leemos las tablas Delta una sola vez para evitar múltiples lecturas a disco durante la ejecución.

In [ ]:
print(f'[{datetime.now()}] Cargando tablas base...')

#Leemos ofertas (solo lo necesario para esta provincia)
df_base_ofertas = (
    tfm_utils.read_from_delta(spark, OFERTAS_TABLE)
    .filter(F.col('provincia') == provincia)
    .filter(F.col('link').isNotNull())
    .select(F.col('link').alias('url'), 'municipio', 'provincia')
    .dropDuplicates(['url'])
    .cache()
)

#Leemos detalles existentes (si existen) para no volver a procesar los que ya hemos guardado
try:
    df_base_detalle = (
        tfm_utils.read_from_delta(spark, DETALLE_TABLE)
        .filter(F.col('provincia') == provincia)
        .select('url', '_scraped_status', 'municipio')
        .cache()
    )
    detalle_existe = True
except Exception:
    df_base_detalle = spark.createDataFrame([], 'url STRING, _scraped_status STRING, municipio STRING')
    detalle_existe = False

print(f'[{datetime.now()}] Carga finalizada.')

[2026-05-04 13:45:54.601344] Cargando tablas base...
[2026-05-04 13:46:16.595544] Carga finalizada.


## Resumen de Progreso General
Calculamos cuántos anuncios han sido ya procesados frente al total de la provincia.

In [4]:
total_urls = df_base_ofertas.count()

if detalle_existe:
    n_ok     = df_base_detalle.filter(F.col('_scraped_status') == 'ok').count()
    n_no_enc = df_base_detalle.filter(F.col('_scraped_status') == 'no_encontrado').count()
    n_err    = df_base_detalle.filter(F.col('_scraped_status') == 'error').count()
    
    progreso_pct = (n_ok + n_no_enc) / total_urls * 100 if total_urls > 0 else 0
    print(f"Progreso en {provincia}: {progreso_pct:.1f}% ({n_ok + n_no_enc} / {total_urls})")
    print(f"Detalle: OK={n_ok} | NO_ENC={n_no_enc} | ERROR={n_err}")
else:
    print(f"Tabla de detalles vacía. Pendiente: {total_urls} URLs.")

Progreso en murcia: 52.6% (7696 / 14644)
Detalle: OK=7050 | NO_ENC=646 | ERROR=906


## Detección de anuncios pendientes
Filtramos las URLs que realmente necesitan ser scrapeadas en esta sesión.

In [ ]:
# Excluimos los URLs ya procesados exitosamente ('ok') o confirmados como desactivados ('no_encontrado')
# Los 'error' se pueden reintentar si RETRY_ERRORS=True
if RETRY_ERRORS:
    # Solo excluimos ok y no_encontrado, los error se reintentan (se hará un left_anti)
    urls_ya_procesadas = df_base_detalle.filter(
        F.col('_scraped_status').isin('ok', 'no_encontrado')
    ).select('url')
else:
    # Excluimos todos los URLs ya procesados (ok, no_encontrado, error)
    urls_ya_procesadas = df_base_detalle.select('url')

df_pendientes = df_base_ofertas.join(urls_ya_procesadas, on='url', how='left_anti')

n_pendientes = df_pendientes.count()
print(f'URLs pendientes de procesar: {n_pendientes} / {total_urls}')
print(datetime.now())

URLs pendientes de procesar: 7128 / 14644
2026-05-04 13:47:14.710350


## Scraping por lotes con guardado incremental
Se procesan `BATCH_SIZE` URLs por lote. Tras cada lote se hace un **MERGE en Delta usando `url` como clave única**, de forma que:
- Si la URL no existe → se inserta.
- Si ya existe (error/no_encontrado) → se actualiza con el nuevo resultado.
- Si el notebook se interrumpe, el siguiente intento retoma desde donde se quedó.

In [6]:
rows_pendientes = df_pendientes.collect()

if debug_mode:
    # En debug: máximo 1 URL por municipio
    seen_mun, debug_rows = set(), []
    for row in rows_pendientes:
        if row['municipio'] not in seen_mun:
            debug_rows.append(row)
            seen_mun.add(row['municipio'])
    rows_pendientes = debug_rows
    print(f'[DEBUG] {len(rows_pendientes)} URLs (1 por municipio)')

pares = [(row['url'], row['municipio']) for row in rows_pendientes]
total_a_procesar = len(pares)
n_batches = math.ceil(total_a_procesar / BATCH_SIZE) if total_a_procesar > 0 else 0

print(f'Total a procesar: {total_a_procesar} URLs en {n_batches} lotes de {BATCH_SIZE}')
print(datetime.now())

Total a procesar: 7128 URLs en 119 lotes de 60
2026-05-04 13:47:15.276457


In [7]:
def merge_lote_en_delta(df_lote_spark, detalle_path, spark):
    """UPSERT del lote en Delta usando 'url' como clave. Crea la tabla si no existe."""
    try:
        dt = DeltaTable.forPath(spark, detalle_path)
        (
            dt.alias('target')
            .merge(df_lote_spark.alias('source'), 'target.url = source.url')
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    except Exception:
        # Primera escritura: la tabla aún no existe
        (
            df_lote_spark
            .write
            .partitionBy('provincia', 'municipio')
            .option('mergeSchema', 'true')
            .mode('append')
            .format('delta')
            .save(detalle_path)
        )

In [ ]:
urls_con_error = []
total_ok, total_no_enc, total_err = 0, 0, 0

for idx in range(0, total_a_procesar, BATCH_SIZE):
    batch_pares = pares[idx : idx + BATCH_SIZE]
    batch_urls  = [p[0] for p in batch_pares]
    url_to_mun  = {p[0]: p[1] for p in batch_pares}
    batch_num   = idx // BATCH_SIZE + 1

    print(f'\n── Lote {batch_num}/{n_batches} · {len(batch_urls)} URLs ──')

    try:
        resultados = idealista.run_async(
            idealista.scrape_property_detail(batch_urls, debug=debug_mode)
        )

        for r in resultados:
            r['provincia']   = provincia
            r['municipio']   = url_to_mun.get(r.get('url', ''), None)
            r['_scraped_at'] = datetime.now().isoformat()
            # Convertir listas a JSON string para compatibilidad con Delta
            for col in ['photo_urls', 'plan_urls', 'caracteristicas_basicas', 'equipamiento', 'cert_energetico']:
                if col in r and isinstance(r[col], list):
                    r[col] = json.dumps(r[col], ensure_ascii=False)

        df_pd   = pd.DataFrame(resultados).astype(str)
        df_pd   = tfm_utils.normalize_df_pandas(df_pd)
        df_lote = spark.createDataFrame(df_pd)

        merge_lote_en_delta(df_lote, detalle_path, spark)

        n_ok  = sum(1 for r in resultados if r.get('_scraped_status') == 'ok')
        n_ne  = sum(1 for r in resultados if r.get('_scraped_status') == 'no_encontrado')
        n_err = sum(1 for r in resultados if r.get('_scraped_status') == 'error')
        total_ok     += n_ok
        total_no_enc += n_ne
        total_err    += n_err
        print(f'  ok={n_ok} | no_encontrado={n_ne} | error={n_err}  → guardado')

    except Exception as e:
        print(f'  ERROR en lote {batch_num}: {e}')
        traceback.print_exc()
        urls_con_error.extend(batch_urls)
        

    if idx + BATCH_SIZE < total_a_procesar:
        time.sleep(SLEEP_BETWEEN_BATCHES)


print(f'\nResumen {provincia}: ok={total_ok} | no_encontrado={total_no_enc} | error={total_err} | lotes_fallidos={len(urls_con_error)}')
print(datetime.now())

## Verificación del resultado

In [9]:
# Liberamos caché para ver resultados frescos de disco
df_base_detalle.unpersist()

df_detalle = (
    tfm_utils.read_from_delta(spark, DETALLE_TABLE)
    .filter(F.col('provincia') == provincia)
)

print(f'Total registros en detalle_propiedades para {provincia}: {df_detalle.count()}')
df_detalle.groupBy('_scraped_status').count().show()

tfm_utils.display(
    df_detalle
    .filter(F.col('_scraped_status') == 'ok')
    .select('url', 'municipio', 'price', 'num_fotos', 'has_price_drop', '_scraped_status'),
    5
)

Total registros en detalle_propiedades para murcia: 15730
+---------------+-----+
|_scraped_status|count|
+---------------+-----+
|             ok|13557|
|          error|  947|
|  no_encontrado| 1226|
+---------------+-----+



,url,municipio,price,num_fotos,has_price_drop,_scraped_status
0,https://www.idealista.com/inmueble/111265019/,murcia,339900.0,42.0,False,ok
1,https://www.idealista.com/inmueble/111265716/,murcia,299500.0,14.0,False,ok
2,https://www.idealista.com/inmueble/111265918/,murcia,265000.0,15.0,False,ok
3,https://www.idealista.com/inmueble/111265921/,murcia,119900.0,27.0,False,ok
4,https://www.idealista.com/inmueble/111265980/,murcia,235000.0,25.0,False,ok


In [10]:
# Si hubo lotes que fallaron completamente, marcamos el proceso como fallido
if urls_con_error:
    raise Exception(
        f'Fallaron {len(urls_con_error)} URLs en {provincia}. '
        f'Primeras 10: {urls_con_error[:10]}'
    )

print(f'[{datetime.now()}] NB3 completado correctamente para {provincia}.')

[2026-05-04 18:31:38.621528] NB3 completado correctamente para murcia.
